In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "I love Transformers!"

# Strategy 1: Character-level
# Output: ['I', ' ', 'l', 'o', 'v', 'e', ...]
# Problem: Too many tokens, loses word meaning

# Strategy 2: Word-level (BERT uses WordPiece)
# Output: ['I', 'love', 'Transform', '##ers', '!']
# Benefit: Unknown words split into subwords
tokens = tokenizer.tokenize(text)
# ['i', 'love', 'transformers', '!']

# Strategy 3: Byte-Pair Encoding (GPT-2)
# Learns merges of frequent character pairs

# Get token IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)
# [1045, 2572, 17953, 999]

# Full encoding with special tokens
encoding = tokenizer(
    text,
    padding="max_length",      # Pad to 512 tokens
    truncation=True,            # Cut if too long
    max_length=512,
    return_tensors="pt"         # Return PyTorch tensors
)

print(encoding.keys())
# dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])

# input_ids: Token IDs
# [101, 1045, 2572, 17953, 999, 102, 0, 0, 0, ...]   padding

# attention_mask: Which tokens to attend to (1=real, 0=padding)
# [1, 1, 1, 1, 1, 1, 0, 0, 0, ...]

# token_type_ids: Sentence identifiers (for sentence pairs)
# [0, 0, 0, 0, 0, 0, 0, 0, 0, ...]

In [2]:
# tokenizing multiple texts
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
texts = [
    "I love transfrmers!",
    "hello world"

]

encodings = tokenizer(
    texts,
    padding = True,
    truncation = True,
    max_length = 512,
    return_tensors = "pt"
)

In [8]:
from transformers import AutoModel,  AutoTokenizer 
import torch 


model = AutoModel.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "hello World"
inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs, output_hidden_states = True)
# outputs.last_hidden_state: Last layer output
# Shape: [batch_size, seq_length, hidden_size]
# [1, 4, 768]

print(outputs.last_hidden_state.shape)
print(outputs.pooler_output.shape)

# outputs.hidden_states: All 13 layers (embedding + 12 transformer)
print(len(outputs.hidden_states))  # 13

# Use for different purposes:
# - Embeddings: Use hidden_states[-1] for sentence-level tasks
# - Token embeddings: Use each token's hidden state separately
# - Layer-wise analysis: Compare how representations change across layers


torch.Size([1, 4, 768])
torch.Size([1, 768])
13


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

model = AutoModel.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

texts = [
    "The cat sat on the mat",
    "A feline was on the rug",
    "transformer is good "
]

# Get embeddings
embeddings = []
for text in texts:
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    # Use pooler output for sentence-level representation
    embedding = outputs.pooler_output.detach().numpy()
    embeddings.append(embedding)

# Compute similarity
similarity = cosine_similarity(embeddings[0], embeddings[1])[0][0]
print(f"Similarity between sentence 1 and 2: {similarity:.4f}")


similarity = cosine_similarity(embeddings[0], embeddings[2])[0][0]
print(f"Similarity between sentence 1 and 3: {similarity:.4f}")

Similarity between sentence 1 and 2: 0.9794
Similarity between sentence 1 and 3: 0.8297
